In [9]:
# Building a sample vector DB with Chroma
# NOTE: the official wrapper is 'from langchain_chroma import Chroma',
# but langchain-chroma is not installed in this venv yet.
# To use it later run:  .\.venv\Scripts\python.exe -m pip install langchain-chroma
# Until then we use the Chroma wrapper that ships inside langchain-community
# (it only prints a deprecation warning, everything works).
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [2]:
# Path is relative to the notebook, so it points at Vector_Store/speech.txt
# (that file exists in this folder). encoding avoids Windows cp1252 read errors.
loader = TextLoader("speech.txt", encoding="utf-8")
documents = loader.load()
documents

[Document(metadata={'source': 'speech.txt'}, page_content="The world must be made safe for democracy. Its peace must be planted upon the\ntested foundations of political liberty. We have no selfish ends to serve. We\ndesire no conquest, no dominion. We seek no indemnities for ourselves, no\nmaterial compensation for the sacrifices we shall freely make.\n\nWe are but one of the champions of the rights of mankind. We shall be satisfied\nwhen those rights have been made as secure as the faith and the freedom of\nnations can make them.\n\nIt is a fearful thing to lead this great peaceful people into war, into the most\nterrible and disastrous of all wars, civilization itself seeming to be in the\nbalance. But the right is more precious than peace, and we shall fight for the\nthings which we have always carried nearest our hearts.\n\nTo such a task we can dedicate our lives and our fortunes, everything that we\nare and everything that we have, with the pride of those who know that the day\n

In [3]:
# Create the embedding object.
# Two mistakes were here: the class was written WITHOUT brackets, which just
# stored the class itself, and no model was named, so it had nothing to call.
# Call it like a function: OllamaEmbeddings(model="...") -> a real object.
embeddings = OllamaEmbeddings(model="mxbai-embed-large")

# loader.load() gave us ONE big Document. Split it so retrieval returns
# small, precise pieces instead of the whole speech every time.
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
chunks = splitter.split_documents(documents)

# Build the vector DB. persist_directory saves it to disk as ./chroma_db
# so the embeddings survive a kernel restart (no need to re-embed).
vector = Chroma.from_documents(documents=chunks,embedding = embeddings)




C:\Users\ap036\AppData\Local\Temp\ipykernel_36280\1395065710.py:5: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="mxbai-embed-large")


In [4]:
query = "what does the speaker say about the democracy and political liberty?"
docs = vector.similarity_search(query)

In [5]:
#for saving to the disk
vectordb = Chroma.from_documents(documents = chunks,embedding = embeddings ,persist_directory="./chroma_db")

In [6]:
# Re-opening a saved DB. This is how you use it in a later session:
#    Chroma(persist_directory=..., embedding_function=...)
#
# THE BUG: this said "./chromadb" but the data was saved to "./chroma_db"
# (chroma_db, with an underscore - see the cell above). Those are two
# DIFFERENT folders. So Chroma happily created a brand new EMPTY database
# in ./chromadb, similarity_search() returned [], and docs[0] blew up with
# IndexError: list index out of range.
#
# Rule: persist_directory is just a folder path. It must be EXACTLY the same
# string you saved to - a missing underscore = a different, empty DB.
db2 = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)

# Quick sanity check BEFORE searching - this is how you catch the bug above:
print("collections:", db2._collection.count())


collections: 55


C:\Users\ap036\AppData\Local\Temp\ipykernel_36280\2971972450.py:12: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db2 = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)


In [7]:
docs = db2.similarity_search(query)

# Guard the index: search can return an empty list if the DB is empty or the
# question has no good match, and docs[0] then raises IndexError.
if docs:
    print(docs[0].page_content)
else:
    print("No matches. Check that persist_directory matches the folder you saved to.")


The world must be made safe for democracy. Its peace must be planted upon the
tested foundations of political liberty. We have no selfish ends to serve. We
desire no conquest, no dominion. We seek no indemnities for ourselves, no
material compensation for the sacrifices we shall freely make.

We are but one of the champions of the rights of mankind. We shall be satisfied
when those rights have been made as secure as the faith and the freedom of
nations can make them.


In [8]:
### retrieber option
retriver = vectordb.as_retriever()
retriver.invoke(query)[0].page_content

'The world must be made safe for democracy. Its peace must be planted upon the\ntested foundations of political liberty. We have no selfish ends to serve. We\ndesire no conquest, no dominion. We seek no indemnities for ourselves, no\nmaterial compensation for the sacrifices we shall freely make.\n\nWe are but one of the champions of the rights of mankind. We shall be satisfied\nwhen those rights have been made as secure as the faith and the freedom of\nnations can make them.'